## Expected result

Algorithm implemented by Yue to find the minimal Steiner tree among input genes.

In [ ]:
import KGBN
gene_list = ['CDKN2A', 'MYC', 'TP53']
bn_string, relations = KGBN.load_signor_network(gene_list)

number of genes found: 3
[1029, 4609, 7157]

Edge found among CDKN2A, MYC, TP53, CDK2:
CDK2 down-regulates CDKN2A
CDK2 up-regulates activity TP53
CDK2 up-regulates CDK2
MYC down-regulates quantity by repression CDKN2A
MYC down-regulates quantity by repression CDKN2A
CDK2 up-regulates quantity by stabilization MYC
TP53 up-regulates quantity by stabilization CDKN2A


# TCT result

In [ ]:
gene_list = ['CDKN2A', 'MYC', 'TP53']
type='gene'
joiner='&'
n_hop=2
# sele_predicate=['biolink:regulates']
sele_predicate=None
sele_API=['CATRAX Pharmacogenomics KP - TRAPI 1.5.0']
sele_KG=['SIGNOR']
sele_tax_id=9606

from TCT import name_resolver, translator_metakg, translator_kpinfo, translator_query, TCT
import igraph as ig

if ' ' not in joiner and joiner in ('&', '|'):
    joiner = ' ' + joiner + ' '

category_map = {
    'gene': 'biolink:Gene',
    'protein': 'biolink:Protein',
    'chemical': 'biolink:ChemicalSubstance',
    'drug': 'biolink:Drug',
    'disease': 'biolink:DiseaseOrPhenotypicFeature',
    'phenotype': 'biolink:PhenotypicFeature',
}
input_node_category = category_map.get(type)
if input_node_category is None:
    raise ValueError(f"Unknown type '{type}'. Must be one of: {list(category_map.keys())}")

# 1. Resolve gene symbols → CURIEs
print("Resolving gene names...")
if sele_tax_id:
    sele_tax_id = "NCBITaxon:" + str(sele_tax_id)
    print(f"Selected tax id: {sele_tax_id}")
else:
    print("No tax id selected, using all")
input_info = name_resolver.batch_lookup(gene_list, only_taxa = sele_tax_id)
input_curies = []
curie_to_symbol = {}
for g in gene_list:
    info = input_info.get(g)
    print(f"info for {g}: {info}")
    if info is not None and hasattr(info, 'curie') and info.curie:
        input_curies.append(info.curie)
        curie_to_symbol[info.curie] = g
print(f"Resolved {len(input_curies)}/{len(gene_list)} genes to CURIEs")
if len(input_curies) < 2:
    print("Need at least 2 resolved genes to build a network.")

# 2. Load Translator resources & determine available predicates/APIs
print("Loading Translator resources...")
_, APInames = translator_kpinfo.get_translator_kp_info()
metaKG = translator_metakg.get_KP_metadata(APInames)
APInames, metaKG = translator_metakg.add_plover_API(APInames, metaKG)

sub_cat = [input_node_category]
obj_cat = [input_node_category]
all_predicates = list(set(
    TCT.select_concept(sub_list=sub_cat, obj_list=obj_cat, metaKG=metaKG)))
all_APIs = TCT.select_API(sub_list=sub_cat, obj_list=obj_cat, metaKG=metaKG)

query_APIs = all_APIs
if sele_API is not None:
    filtered_APIs = [a for a in all_APIs if a in sele_API]
    if filtered_APIs:
        query_APIs = filtered_APIs
    else:
        print(f"Warning: none of sele_API found. Using all {len(all_APIs)} available APIs.")

API_predicates = {}
for api in set(metaKG['API']):
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

# 3. Query Translator
def _do_query(curies):
    qj = TCT.format_query_json(curies, [], sub_cat, obj_cat, all_predicates)
    return translator_query.parallel_api_query(
        query_json=qj, select_APIs=query_APIs,
        APInames=APInames, API_predicates=API_predicates,
        max_workers=len(query_APIs))

print(f"Querying Translator with {len(input_curies)} input nodes...")
all_results = _do_query(input_curies)
print(f"Query returned {len(all_results)} results")


Resolving gene names...
Selected tax id: NCBITaxon:9606
info for CDKN2A: TranslatorNode(curie='NCBIGene:1029', label='CDKN2A', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachineMixin', 'biolink:Protein', 'biolink:GeneProductMixin', 'biolink:Polypeptide', 'biolink:ChemicalEntityOrProteinOrPolypeptide'], synonyms=None, curie_synonyms=None, attributes=None, taxa=['NCBITaxon:9606'])
info for MYC: TranslatorNode(curie='NCBIGene:4609', label='MYC', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity',

In [ ]:
# 5. Filter results by KG
sele_KG_set = set(sele_KG) if sele_KG else None

filtered = {}
for k, v in all_results.items():
    if not isinstance(v, dict):
        continue
    if sele_KG_set:
        match = False
        for src in v.get('sources', []):
            rid = src.get('resource_id', '')
            if rid in sele_KG_set or rid.split(':')[-1] in sele_KG_set:
                match = True
                break
        if not match:
            for attr in v.get('attributes', []):
                if attr.get('attribute_type_id') in ('provided_by', 'knowledge_source'):
                    if attr.get('value') in sele_KG_set:
                        match = True
                        break
        if not match:
            continue
    filtered[k] = v

len(filtered)

189

In [ ]:
name_resolver.batch_lookup(['CDK2'], only_taxa = sele_tax_id)

{'CDK2': TranslatorNode(curie='NCBIGene:1017', label='CDK2', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachineMixin', 'biolink:Protein', 'biolink:GeneProductMixin', 'biolink:Polypeptide', 'biolink:ChemicalEntityOrProteinOrPolypeptide'], synonyms=None, curie_synonyms=None, attributes=None, taxa=['NCBITaxon:9606'])}

In [ ]:
for k, v in filtered.items():
    if v['subject'] in ['NCBIGene:1029','NCBIGene:4609','NCBIGene:7157','NCBIGene:1017']:
        if v['object'] in ['NCBIGene:1029','NCBIGene:4609','NCBIGene:7157','NCBIGene:1017']:
            print(f"{v['subject']} {v['predicate']} {v['object']}")

NCBIGene:1029 biolink:regulates NCBIGene:7157
NCBIGene:4609 biolink:regulates NCBIGene:1029
NCBIGene:1017 biolink:regulates NCBIGene:7157


Retrieved edges:  
CDKN2A, TP53  
MYC, CDKN2A  
CDK2, TP53  

Compare with expected:  
CDK2 down-regulates CDKN2A  
CDK2 up-regulates activity TP53  
CDK2 up-regulates CDK2  
MYC down-regulates quantity by repression CDKN2A  
MYC down-regulates quantity by repression CDKN2A  
CDK2 up-regulates quantity by stabilization MYC  
TP53 up-regulates quantity by stabilization CDKN2A  

I confirmed that they exist in searching on SIGNOR directly: https://signor.uniroma2.it/relation_result.php?id=P24941

Qi's additional code to check on parsed SIGNOR source file  
Identifier:   
1. CDK2 down-regulates CDKN2A  
CDK2: NCBIGene:1017, CDKN2A: NCBIGene:1029  

2. CDK2 up-regulates activity TP53  
CDK2: NCBIGene:1017, TP53: NCBIGene:7157  

3. CDK2 up-regulates CDK2  
CDK2: NCBIGene:1017  

5. MYC down-regulates quantity by repression CDKN2A
MYC: NCBIGene:4609, CDKN2A: NCBIGene:1029  

6. CDK2 up-regulates quantity by stabilization MYC
CDK2: NCBIGene:1017, MYC: NCBIGene:4609  

7. TP53 up-regulates quantity by stabilization CDKN2A
TP53: NCBIGene:7157, CDKN2A: NCBIGene:1029  

In [3]:
## Load necessary packages
import os
import pandas as pd
import glob
import numpy as np
import networkx as nx
# import matplotlib.pyplot as plt

## load the needed tqdm package
from tqdm import tqdm

# Enable tqdm for pandas apply
tqdm.pandas()

In [5]:
## Notice!! Please change the file path of following codes into your own
parsed_files_path = '/Users/Weiqi0/ISB_working/Ilya_lab/Translator/Pharmagenomics_KG/files/parsed/wQualityControl/'

## Note! define download path in the very end of the notebook!!

## Read each individual csv files
source_df = pd.read_csv(parsed_files_path + 'Pharmacogenomics_KG_parsed_edge_09_04_2025.tsv', sep='\t')

In [6]:
source_df.head(10)

,subject,subject_name,subject_category,object,object_name,object_category,predicate,knowledge_source,knowledge_level,context_qualifier,agent_type,deploy_date,id,publications,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier,provided_by
0,RXCUI:2563966,Odevixibat,biolink:Drug,NCBIGene:8647,ABCB11,biolink:Gene,biolink:has_biomarker,FDA pharmacogenomics biomarkers table,knowledge_assertion,Gastroenterology,text_mining_agent,2025-09-04,4e1f7def-7e8d-5204-9826-56887ff5ecb5,NaN,NaN,NaN,NaN,NaN
1,RXCUI:2571074,Maralixibat,biolink:Drug,NCBIGene:8647,ABCB11,biolink:Gene,biolink:has_biomarker,FDA pharmacogenomics biomarkers table,knowledge_assertion,Gastroenterology,text_mining_agent,2025-09-04,d5f51f86-e8a1-5481-8105-cdd2d9efc023,NaN,NaN,NaN,NaN,NaN
2,RXCUI:2571074,Maralixibat,biolink:Drug,NCBIGene:5244,ABCB4,biolink:Gene,biolink:has_biomarker,FDA pharmacogenomics biomarkers table,knowledge_assertion,Gastroenterology,text_mining_agent,2025-09-04,ef3537de-0846-5bf9-b22a-5541c4214210,NaN,NaN,NaN,NaN,NaN
3,RXCUI:2677894,Resmetirom,biolink:Drug,NCBIGene:9429,ABCG2,biolink:Gene,biolink:has_biomarker,FDA pharmacogenomics biomarkers table,knowledge_assertion,Gastroenterology,text_mining_agent,2025-09-04,871b4518-e7db-5e9b-b5f8-c3ec2e4eed69,NaN,NaN,NaN,NaN,NaN
4,RXCUI:1313234,Triheptanoin,biolink:Drug,MONDO:0008723,"ACADVL, CPT2, HADHA, HADHB (Long-Chain Fatty A...",biolink:Disease,biolink:has_biomarker,FDA pharmacogenomics biomarkers table,knowledge_assertion,Inborn Errors of Metabolism,text_mining_agent,2025-09-04,92c91a1c-5cee-5fac-9962-a15a5800821b,NaN,NaN,NaN,NaN,NaN
5,RXCUI:2672492,Zilucoplan,biolink:Drug,NCBIGene:1134,ACHR,biolink:Gene,biolink:has_biomarker,FDA pharmacogenomics biomarkers table,knowledge_assertion,Neurology,text_mining_agent,2025-09-04,7bde8aa1-e508-5d70-b000-f779050e4c7c,NaN,NaN,NaN,NaN,NaN
6,RXCUI:2642273,Rozanolixizumab-noli,biolink:Drug,NCBIGene:1134,ACHR,biolink:Gene,biolink:has_biomarker,FDA pharmacogenomics biomarkers table,knowledge_assertion,Neurology,text_mining_agent,2025-09-04,0f7f7a8e-a467-526b-a065-44d0eeb0508a,NaN,NaN,NaN,NaN,NaN
7,RXCUI:2107316,Ravulizumab-cwvz,biolink:Drug,NCBIGene:1134,ACHR,biolink:Gene,biolink:has_biomarker,FDA pharmacogenomics biomarkers table,knowledge_assertion,Neurology,text_mining_agent,2025-09-04,8a61035a-c46d-5e33-a1b3-4adbc8c4272e,NaN,NaN,NaN,NaN,NaN
8,RXCUI:2587719,Efgartigimod Alfa-fcab,biolink:Drug,NCBIGene:1134,ACHR,biolink:Gene,biolink:has_biomarker,FDA pharmacogenomics biomarkers table,knowledge_assertion,Neurology,text_mining_agent,2025-09-04,8514da8f-7438-5365-b82e-4d82efb5cdb7,NaN,NaN,NaN,NaN,NaN
9,RXCUI:591781,Eculizumab,biolink:Drug,NCBIGene:1134,ACHR,biolink:Gene,biolink:has_biomarker,FDA pharmacogenomics biomarkers table,knowledge_assertion,Neurology,text_mining_agent,2025-09-04,f1438026-4ccf-5a8d-adc2-b5c5c24abb3b,NaN,NaN,NaN,NaN,NaN


1. CDK2 down-regulates CDKN2A  
In SIGNOR web portal results: I only saw CDKN2A -> CDK2?

In [7]:
filtered_df = source_df[(source_df['subject'] == 'NCBIGene:1017') & (source_df['object'] == 'NCBIGene:1029')]

filtered_df

,subject,subject_name,subject_category,object,object_name,object_category,predicate,knowledge_source,knowledge_level,context_qualifier,agent_type,deploy_date,id,publications,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier,provided_by


In [11]:
filtered_df = source_df[(source_df['subject'] == 'NCBIGene:1029') & (source_df['object'] == 'NCBIGene:1017')]

filtered_df

,subject,subject_name,subject_category,object,object_name,object_category,predicate,knowledge_source,knowledge_level,context_qualifier,agent_type,deploy_date,id,publications,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier,provided_by
44138,NCBIGene:1029,CDKN2A,biolink:Gene,NCBIGene:1017,CDK2,biolink:Gene,biolink:regulates,SIGNOR-64431,knowledge_assertion,NaN,automated_agent,2025-08-18,40f8be1c-6547-5c2d-9c31-f8a5dc55ae1c,PMID:36243968|PMID:10022885,NaN,downregulated,NaN,SIGNOR


In [ ]:
2. CDK2 up-regulates activity TP53

In [8]:
filtered_df = source_df[(source_df['subject'] == 'NCBIGene:1017') & (source_df['object'] == 'NCBIGene:7157')]

filtered_df

,subject,subject_name,subject_category,object,object_name,object_category,predicate,knowledge_source,knowledge_level,context_qualifier,agent_type,deploy_date,id,publications,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier,provided_by
27692,NCBIGene:1017,CDK2,biolink:Gene,NCBIGene:7157,TP53,biolink:Gene,biolink:regulates,SIGNOR-119379,knowledge_assertion,NaN,automated_agent,2025-08-18,f3c8aa3b-1107-5028-b474-b47a6fd9d595,PMID:36243968|PMID:24173284,NaN,upregulated,NaN,SIGNOR


3. CDK2 up-regulates CDK2 

In [9]:
filtered_df = source_df[(source_df['subject'] == 'NCBIGene:1017') & (source_df['object'] == 'NCBIGene:1017')]

filtered_df

,subject,subject_name,subject_category,object,object_name,object_category,predicate,knowledge_source,knowledge_level,context_qualifier,agent_type,deploy_date,id,publications,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier,provided_by


5. MYC down-regulates quantity by repression CDKN2A
MYC: NCBIGene:4609, CDKN2A: NCBIGene:1029 

In [10]:
filtered_df = source_df[(source_df['subject'] == 'NCBIGene:4609') & (source_df['object'] == 'NCBIGene:1029')]

filtered_df

,subject,subject_name,subject_category,object,object_name,object_category,predicate,knowledge_source,knowledge_level,context_qualifier,agent_type,deploy_date,id,publications,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier,provided_by
21679,NCBIGene:4609,MYC,biolink:Gene,NCBIGene:1029,CDKN2A,biolink:Gene,biolink:regulates,SIGNOR-267575,knowledge_assertion,NaN,automated_agent,2025-08-18,9849d9e2-b6d7-58b7-a1c6-3aaee93f14cf,PMID:36243968|PMID:20551174,NaN,downregulated,NaN,SIGNOR
24728,NCBIGene:4609,MYC,biolink:Gene,NCBIGene:1029,CDKN2A,biolink:Gene,biolink:regulates,SIGNOR-102743,knowledge_assertion,NaN,automated_agent,2025-08-18,9849d9e2-b6d7-58b7-a1c6-3aaee93f14cf,PMID:36243968|PMID:12835716,NaN,downregulated,NaN,SIGNOR


6. CDK2 up-regulates quantity by stabilization MYC
CDK2: NCBIGene:1017, MYC: NCBIGene:4609  

In [12]:
filtered_df = source_df[(source_df['subject'] == 'NCBIGene:1017') & (source_df['object'] == 'NCBIGene:4609')]

filtered_df

,subject,subject_name,subject_category,object,object_name,object_category,predicate,knowledge_source,knowledge_level,context_qualifier,agent_type,deploy_date,id,publications,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier,provided_by


In [13]:
filtered_df = source_df[(source_df['subject'] == 'NCBIGene:4609') & (source_df['object'] == 'NCBIGene:1017')]

filtered_df

,subject,subject_name,subject_category,object,object_name,object_category,predicate,knowledge_source,knowledge_level,context_qualifier,agent_type,deploy_date,id,publications,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier,provided_by


7. TP53 up-regulates quantity by stabilization CDKN2A
TP53: NCBIGene:7157, CDKN2A: NCBIGene:1029  

In [14]:
filtered_df = source_df[(source_df['subject'] == 'NCBIGene:7157') & (source_df['object'] == 'NCBIGene:1029')]

filtered_df

,subject,subject_name,subject_category,object,object_name,object_category,predicate,knowledge_source,knowledge_level,context_qualifier,agent_type,deploy_date,id,publications,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier,provided_by


In [15]:
filtered_df = source_df[(source_df['subject'] == 'NCBIGene:1029') & (source_df['object'] == 'NCBIGene:7157')]

filtered_df

,subject,subject_name,subject_category,object,object_name,object_category,predicate,knowledge_source,knowledge_level,context_qualifier,agent_type,deploy_date,id,publications,subject_direction_qualifier,object_direction_qualifier,object_aspect_qualifier,provided_by
26817,NCBIGene:1029,CDKN2A,biolink:Gene,NCBIGene:7157,TP53,biolink:Gene,biolink:regulates,SIGNOR-255694,knowledge_assertion,NaN,automated_agent,2025-08-18,6d84caa9-98ae-559f-afcb-009091d7c4f8,PMID:36243968|PMID:12091906,NaN,upregulated,NaN,SIGNOR
